# Figure S5 — Dendrograms of language clusters

Hierarchical clustering (Ward's method) applied to 28 languages based on their
within-language and between-language expression vectors over time.
Six variants are saved: WL log, WL non-log, BL log, BL non-log, raw freq, raw freq log.

**Inputs:** `CHOSEN_WEEKLY_PIVOT_FILE` (`data/processed/chosen_words_weekly_pivoted.csv`)
**Outputs:** `outputs/figures/Fig.S5_dendrograms/`
**Prerequisites:** run `02_combine_data.ipynb` first

In [ ]:
import sys
sys.path.insert(0, '..')
from config import CHOSEN_WEEKLY_PIVOT_FILE, WORD_FORMS_ALL, FIGURES_DIR

import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import dendrogram, linkage

## ⚙️ Parameters

In [ ]:
remove_UA_RU = False
smooth       = False
smooth_time  = 6  # weeks of rolling-mean smoothing (used only when smooth=True)

## Load & compute expression

In [ ]:
DFfreq = pd.read_csv(CHOSEN_WEEKLY_PIVOT_FILE, index_col=0)

# Robust index handling: supports both ISO-index and language-index pivots
meta_lang = pd.read_csv(WORD_FORMS_ALL, usecols=['ISO', 'Language']).drop_duplicates()
iso_to_name = meta_lang.set_index('ISO')['Language'].to_dict()
iso_set = set(meta_lang['ISO'])
name_set = set(meta_lang['Language'])

idx = pd.Index(DFfreq.index.astype(str))
if idx.isin(iso_set).all():
    DFfreq.index = idx.map(iso_to_name)
elif idx.isin(name_set).all():
    DFfreq.index = idx
else:
    DFfreq.index = idx.map(iso_to_name).fillna(idx)
DFfreq.index = pd.Index(DFfreq.index, name='language')

if remove_UA_RU:
    DFfreq = DFfreq.drop(index=['Ukrainian', 'Russian'], errors='ignore')

# ── Within-language expression ────────────────────────────────────────────────
freqRowMeans = DFfreq.mean(axis=1)
safeFreqRowMeans = freqRowMeans.replace(0, np.nan)
WLDFresultNonLog = DFfreq.sub(safeFreqRowMeans, axis=0).div(safeFreqRowMeans, axis=0)
WLDFresultNonLog = WLDFresultNonLog.replace([np.inf, -np.inf], np.nan)
WLDFresultLog = WLDFresultNonLog.map(
    lambda x: 0 if pd.isna(x) or x == 0 else (np.log10(x * 100_000) if x > 0 else -np.log10(-x * 100_000))
)

# ── Between-language expression ───────────────────────────────────────────────
SumsFreq = DFfreq.sum().replace(0, np.nan)
DFfreqShare = DFfreq.div(SumsFreq, axis=1)
safe_total_mean = freqRowMeans.sum()
freqShareRowMeans_Proportion = (
    freqRowMeans / safe_total_mean if safe_total_mean != 0 else freqRowMeans * np.nan
)
BLDFresultNonLog = DFfreqShare.sub(freqShareRowMeans_Proportion, axis=0)
BLDFresultNonLog = BLDFresultNonLog.replace([np.inf, -np.inf], np.nan)
BLDFresultLog = BLDFresultNonLog.map(
    lambda x: 0 if pd.isna(x) or x == 0 else (np.log10(x * 100_000) if x > 0 else -np.log10(-x * 100_000))
)

# ── Log-transform of raw frequencies ─────────────────────────────────────────
DFfreqLog = DFfreq.map(
    lambda x: 0 if pd.isna(x) or x == 0 else (np.log10(x * 100_000) if x > 0 else -np.log10(-x * 100_000))
)

# ── Optional smoothing ────────────────────────────────────────────────────────
if smooth:
    min_periods = 1
    WLDFresultLog    = WLDFresultLog.T.rolling(window=smooth_time, min_periods=min_periods).mean().T
    WLDFresultNonLog = WLDFresultNonLog.T.rolling(window=smooth_time, min_periods=min_periods).mean().T
    BLDFresultLog    = BLDFresultLog.T.rolling(window=smooth_time, min_periods=min_periods).mean().T
    BLDFresultNonLog = BLDFresultNonLog.T.rolling(window=smooth_time, min_periods=min_periods).mean().T

print("Within-language expression (log) shape:", WLDFresultLog.shape)
print("Between-language expression (log) shape:", BLDFresultLog.shape)

## Quick preview dendrogram

In [ ]:
# Quick preview — within-language log expression
Z_preview = linkage(WLDFresultLog, method='ward')
plt.figure(figsize=(3.46457, 1))
dendrogram(Z_preview, labels=list(WLDFresultLog.index), leaf_rotation=90)
plt.ylabel("Distance")
plt.xticks(fontsize=8, rotation=90)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

## Save dendrograms

In [ ]:
def save_dendrogram(df, name, method='ward', dpi=500, figsize=(7, 3)):
    """
    Compute and save a dendrogram for any DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        Rows are observations to cluster; index labels appear on the leaves.
    name : str
        Short label included in the output filename.
    method : str
        Linkage method (default 'ward').
    dpi : int
        Resolution for the saved PNG (default 500).
    figsize : tuple
        Figure size in inches (default (7, 3)).

    Returns
    -------
    dict with 'png' and 'svg' absolute file paths.
    """
    date_str = datetime.now().strftime("%Y-%m-%d")
    Z = linkage(df, method=method)

    plt.figure(figsize=figsize)
    dendrogram(Z, labels=list(df.index), leaf_rotation=90)
    plt.ylabel("Distance")
    plt.tight_layout()

    out_dir = FIGURES_DIR / "Fig.S5_dendrograms"
    out_dir.mkdir(parents=True, exist_ok=True)

    base     = out_dir / f"dendrogram_{name}_{date_str}"
    png_file = str(base) + ".png"
    svg_file = str(base) + ".svg"

    plt.savefig(png_file, format="png", dpi=dpi)
    plt.savefig(svg_file, format="svg")
    plt.show()

    print(f"Saved PNG: {png_file}")
    print(f"Saved SVG: {svg_file}")
    return {"png": png_file, "svg": svg_file}

In [ ]:
save_dendrogram(WLDFresultLog,    'WL_Log')
save_dendrogram(WLDFresultNonLog, 'WL_notLog')

save_dendrogram(BLDFresultLog,    'BL_Log')
save_dendrogram(BLDFresultNonLog, 'BL_notLog')

save_dendrogram(DFfreq,    'vanillaRelfreq')
save_dendrogram(DFfreqLog, 'vanillaRelfreqLog')